In [1]:
import warnings 
warnings.filterwarnings('ignore')

In [3]:
from peft import LoraConfig , get_peft_model 

# Load the Base Model 
from transformers import AutoTokenizer , AutoModelForCausalLM 

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 10831.00it/s]


In [7]:
# configuration for LoW Rank adaptation 

lora_config = LoraConfig(
    r=8, # size of lora metrics 
    lora_alpha=16 ,# scaling factor 
    lora_dropout=0.05, # dropout for lora layers
    target_modules=["c_attn"], # where lora is added 
    task_type="CAUSAL_LM" # GPT style text generation 
)

model = get_peft_model(model,lora_config)
model.print_trainable_parameters()

trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


In [14]:
from transformers import Trainer , TrainingArguments 
from Supervised_Fine_Tuned import tokenize_dataset

from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="dataset.json"
)
training_arguments = TrainingArguments(
    output_dir="./lora-output",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    learning_rate=5e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model = model ,
    args=training_arguments,
    train_dataset=tokenize_dataset['train']
)

trainer.train()

Step,Training Loss
1,5.020514
2,4.412972
3,5.379876
4,5.185194
5,4.936326
6,4.086915
7,5.062190
8,3.939130
9,4.842793
10,3.860324


TrainOutput(global_step=30, training_loss=4.411306103070577, metrics={'train_runtime': 3.6032, 'train_samples_per_second': 8.326, 'train_steps_per_second': 8.326, 'total_flos': 983260200960.0, 'train_loss': 4.411306103070577, 'epoch': 10.0})

In [15]:
model.save_pretrained("./lora-adapter")
tokenizer.save_pretrained("./lora-adapter")


('./lora-adapter/tokenizer_config.json', './lora-adapter/tokenizer.json')

In [17]:
model.eval()

import torch 

prompt = "Question : what is python?\nanswer :"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=30
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

Question : what is python?
answer : python is a Python programming language. It is a programming language that is written in Python. It is a programming language that is written in Python. It
